In [3]:

from dotenv import load_dotenv
from langgraph.graph import START, END, StateGraph
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from typing import Optional
from typing_extensions import TypedDict
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from IPython.display import Image

load_dotenv()
model = ChatHuggingFace(llm=HuggingFaceEndpoint(model="deepseek-ai/DeepSeek-V3.2"))

# 1. Define the State
# 2. Create a StateGraph
# 3. Add nodes(functions) to the graph
# 4. Add edges to the graph
# 5. Compile the graph
# 6. execute the graph

# Define the State

class State(TypedDict):
    topic: str
    report: str
    summary: str

/home/rkroy/Desktop/code/python/venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


None of PyTorch, TensorFlow >= 2.0, or Flax have been found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [4]:
# Create a StateGraph

graph = StateGraph(State)

In [5]:
# Define all the nodes and add them to the graph

def generate_report(state: State) -> State:
    prompt = PromptTemplate(
        template="write me a brief report on topic: {topic}",
        input_variables=["topic"],
    )
    parser = StrOutputParser()
    chain = prompt | model | parser
    res = chain.invoke({"topic": state["topic"]})
    state["report"] = res
    return state

def generate_summary(state: State) -> State:
    prompt = PromptTemplate(
        template="write me a complete summary of the report: {report}",
        input_variables=["report"],
    )
    parser = StrOutputParser()
    chain = prompt | model | parser
    res = chain.invoke({"report": state["report"]})
    state["summary"] = res
    return state

# Add the nodes to the graph

graph.add_node("generate_report", generate_report)
graph.add_node("generate_summary", generate_summary)

In [6]:
# Add edges to the graph

graph.add_edge(START, "generate_report")
graph.add_edge("generate_report", "generate_summary")
graph.add_edge("generate_summary", END)

In [7]:
# Compile the graph

app = graph.compile()

In [8]:
# Execute the graph

final_state = app.invoke(State({"topic": "python", "report": "", "summary": ""}))

print(final_state)

Image(app.get_graph().draw_mermaid_png())